# 9 Funcion preprocesado

Notebook autocontenido para preprocesar una pareja H&E + HSI.

La funcion principal recibe:

- `he_path`: ruta de la imagen H&E `.mrxs`.
- `hsi_hdr_path`: ruta de la imagen HSI `.hdr`.

Todas las salidas visuales/textuales se pueden activar o desactivar con flags `True/False`.

In [ ]:
from pathlib import Path
from contextlib import redirect_stdout
import io

import cv2
import matplotlib.pyplot as plt
from matplotlib.colors import rgb_to_hsv
import numpy as np
import openslide
from PIL import Image
from scipy import ndimage as ndi

plt.rcParams['figure.dpi'] = 120


## Rutas de ejemplo

In [ ]:
SPECIMENS = {
    'SB012': {
        'he_path': Path(r'Datos\SB012\SB012\H&E\SB012\SB012.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB012\SB012\SB012 HSI\SB012_M1_002\SB012_M1_nrm_edu.hdr'),
    },
    'SB013': {
        'he_path': Path(r'Datos\SB013\SB013\H&E\SB013.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB013\SB013\SB013_001\SB013_nrm_edu.hdr'),
    },
    'SB017': {
        'he_path': Path(r'Datos\SB017_uomo\SB017_uomo\H&E\SB017\SB017.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB017_uomo\SB017_uomo\SB017_001\SB017_nrm_edu.hdr'),
    },
    'SB018': {
        'he_path': Path(r'Datos\SB018\SB018\H&E\SB018\SB018.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB018\SB018\SB018_001\SB018_nrm_edu.hdr'),
    },
    'SB019': {
        'he_path': Path(r'Datos\SB019\SB019\H&E\SB019\SB019.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB019\SB019\SB019_001\SB019_nrm_edu.hdr'),
    },
    'SB020': {
        'he_path': Path(r'Datos\SB020\SB020\H&E\SB020\SB020.mrxs'),
        'hsi_hdr_path': Path(r'Datos\SB020\SB020\SB020_001\SB020_nrm_edu.hdr'),
    },
}

def check_specimen_paths(specimens=SPECIMENS):
    for specimen_id, paths in specimens.items():
        print(specimen_id)
        print('  H&E:', paths['he_path'].exists(), '|', paths['he_path'])
        print('  HSI:', paths['hsi_hdr_path'].exists(), '|', paths['hsi_hdr_path'])



## Funciones HSI: lectura, pseudo-RGB y caja azul

In [ ]:
def parse_envi_header(hdr_path):
    hdr_path = Path(hdr_path)
    text = hdr_path.read_text(encoding='utf-8', errors='ignore')
    metadata = {}
    current_key = None
    current_value = []

    for raw_line in text.splitlines():
        line = raw_line.strip()
        if not line or line.upper() == 'ENVI':
            continue

        if current_key is not None:
            current_value.append(line)
            if '}' in line:
                metadata[current_key] = ' '.join(current_value)
                current_key = None
                current_value = []
            continue

        if '=' not in line:
            continue

        key, value = line.split('=', 1)
        key = key.strip().lower()
        value = value.strip()

        if value.startswith('{') and '}' not in value:
            current_key = key
            current_value = [value]
        else:
            metadata[key] = value

    return metadata


def parse_wavelengths(metadata):
    values = metadata.get('wavelength')
    if values is None:
        return None

    values = values.strip().strip('{}')
    return np.array([float(v.strip()) for v in values.split(',') if v.strip()], dtype=np.float32)


def envi_dtype(metadata):
    data_type = int(metadata['data type'])
    byte_order = int(metadata.get('byte order', 0))
    endian = '<' if byte_order == 0 else '>'
    dtype_map = {
        1: 'u1',
        2: 'i2',
        3: 'i4',
        4: 'f4',
        5: 'f8',
        12: 'u2',
        13: 'u4',
        14: 'i8',
        15: 'u8',
    }
    if data_type not in dtype_map:
        raise ValueError(f'Tipo de dato ENVI no soportado: {data_type}')
    return np.dtype(endian + dtype_map[data_type])


def find_envi_data_path(hdr_path, metadata):
    hdr_path = Path(hdr_path)
    candidates = []

    for key in ('data file', 'file name', 'image'):
        if key in metadata:
            value = metadata[key].strip().strip('{}').strip('"')
            candidates.append(hdr_path.parent / value)

    candidates.extend([
        hdr_path.with_suffix(''),
        hdr_path.parent / hdr_path.stem.replace('_edu', ''),
    ])

    for candidate in candidates:
        if candidate.exists():
            return candidate

    raise FileNotFoundError(f'No se encontro el binario ENVI asociado a {hdr_path.name}')


def read_envi_band(hdr_path, metadata, band_idx):
    data_path = find_envi_data_path(hdr_path, metadata)
    samples = int(metadata['samples'])
    lines = int(metadata['lines'])
    bands = int(metadata['bands'])
    offset = int(metadata.get('header offset', 0))
    interleave = metadata.get('interleave', 'bsq').strip().lower()
    dtype = envi_dtype(metadata)

    if interleave == 'bsq':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(bands, lines, samples))
        return np.asarray(cube[band_idx], dtype=np.float32)
    if interleave == 'bil':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, bands, samples))
        return np.asarray(cube[:, band_idx, :], dtype=np.float32)
    if interleave == 'bip':
        cube = np.memmap(data_path, dtype=dtype, mode='r', offset=offset, shape=(lines, samples, bands))
        return np.asarray(cube[:, :, band_idx], dtype=np.float32)

    raise ValueError(f'Interleave ENVI no soportado: {interleave}')


def robust_normalize(channel, p_low=2, p_high=98):
    channel = np.asarray(channel, dtype=np.float32)
    if not np.any(np.isfinite(channel)):
        return np.zeros_like(channel, dtype=np.float32)

    lo = np.nanpercentile(channel, p_low)
    hi = np.nanpercentile(channel, p_high)

    if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
        return np.zeros_like(channel, dtype=np.float32)

    normalized = np.clip((channel - lo) / (hi - lo), 0, 1)
    return np.nan_to_num(normalized, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)


def hsi_pseudo_rgb_from_path(hdr_path, r_nm=650, g_nm=550, b_nm=450):
    hdr_path = Path(hdr_path)
    if not hdr_path.exists():
        raise FileNotFoundError(f'No existe la HSI: {hdr_path}')

    metadata = parse_envi_header(hdr_path)
    wavelengths = parse_wavelengths(metadata)

    if wavelengths is None:
        band_count = int(metadata['bands'])
        r_idx = min(band_count - 1, int(round(0.72 * (band_count - 1))))
        g_idx = min(band_count - 1, int(round(0.50 * (band_count - 1))))
        b_idx = min(band_count - 1, int(round(0.28 * (band_count - 1))))
        band_info = {'r_idx': r_idx, 'g_idx': g_idx, 'b_idx': b_idx}
    else:
        r_idx = int(np.argmin(np.abs(wavelengths - r_nm)))
        g_idx = int(np.argmin(np.abs(wavelengths - g_nm)))
        b_idx = int(np.argmin(np.abs(wavelengths - b_nm)))
        band_info = {
            'r_idx': r_idx,
            'r_nm': float(wavelengths[r_idx]),
            'g_idx': g_idx,
            'g_nm': float(wavelengths[g_idx]),
            'b_idx': b_idx,
            'b_nm': float(wavelengths[b_idx]),
        }

    rgb = np.stack(
        [
            robust_normalize(read_envi_band(hdr_path, metadata, r_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, g_idx)),
            robust_normalize(read_envi_band(hdr_path, metadata, b_idx)),
        ],
        axis=-1,
    )
    rgb = np.nan_to_num(rgb, nan=0.0, posinf=1.0, neginf=0.0)
    rgb = (np.clip(rgb, 0, 1) * 255).astype(np.uint8)

    info = {
        'shape': rgb.shape,
        'data_path': find_envi_data_path(hdr_path, metadata),
        **band_info,
    }
    return rgb, info



In [ ]:
def draw_rectangle_rgb(img_rgb, bbox, color=(255, 255, 0), thickness=4):
    out = img_rgb.copy()
    x1, y1, x2, y2 = bbox
    h, w = out.shape[:2]
    x1 = int(np.clip(x1, 0, w - 1))
    x2 = int(np.clip(x2, 0, w - 1))
    y1 = int(np.clip(y1, 0, h - 1))
    y2 = int(np.clip(y2, 0, h - 1))
    t = max(1, int(thickness))

    out[y1:y1 + t, x1:x2 + 1] = color
    out[max(y2 - t + 1, 0):y2 + 1, x1:x2 + 1] = color
    out[y1:y2 + 1, x1:x1 + t] = color
    out[y1:y2 + 1, max(x2 - t + 1, 0):x2 + 1] = color
    return out


def overlay_mask_rgb(img_rgb, mask, color=(0, 255, 255), alpha=0.35):
    out = img_rgb.copy().astype(np.float32)
    color_arr = np.array(color, dtype=np.float32)
    mask = mask.astype(bool)
    out[mask] = (1 - alpha) * out[mask] + alpha * color_arr
    return np.clip(out, 0, 255).astype(np.uint8)


def detect_blue_box_from_rgb(
    rgb_u8,
    hue_min=0.48,
    hue_max=0.66,
    sat_min=0.10,
    value_min=0.10,
    value_max=0.90,
    blue_red_min=0.090,
    green_red_min=-0.005,
    blue_excess_min=0.075,
    white_sat_max=0.080,
    white_value_min=0.82,
    open_size=3,
    close_size=25,
    dilate_size=7,
    min_area=1200,
    padding=20,
):
    """Detecta la caja azul de la pseudo-RGB y devuelve mascara, bbox y overlay."""
    rgb_u8 = np.asarray(rgb_u8, dtype=np.uint8)
    rgb = rgb_u8.astype(np.float32) / 255.0
    hsv = rgb_to_hsv(rgb)

    h = hsv[:, :, 0]
    s = hsv[:, :, 1]
    v = hsv[:, :, 2]
    r = rgb[:, :, 0]
    g = rgb[:, :, 1]
    b = rgb[:, :, 2]

    blue_hue = (h >= hue_min) & (h <= hue_max)
    blue_red = b - r
    green_red = g - r
    blue_excess = 0.65 * blue_red + 0.35 * green_red
    blue_dominance = (blue_red > blue_red_min) & (green_red > green_red_min) & (blue_excess > blue_excess_min)
    not_extreme = (v >= value_min) & (v <= value_max)
    not_white = ~((v > white_value_min) & (s < white_sat_max))
    mask = blue_hue & blue_dominance & (s >= sat_min) & not_extreme & not_white

    if open_size > 1:
        mask = ndi.binary_opening(mask, structure=np.ones((open_size, open_size), dtype=bool))
    if close_size > 1:
        mask = ndi.binary_closing(mask, structure=np.ones((close_size, close_size), dtype=bool))
    if dilate_size > 1:
        mask = ndi.binary_dilation(mask, structure=np.ones((dilate_size, dilate_size), dtype=bool))

    labels, num_labels = ndi.label(mask)
    if num_labels == 0:
        raise ValueError('No se detecto ninguna region azul. Ajusta los umbrales HSV/dominancia.')

    areas = np.bincount(labels.ravel())
    areas[0] = 0
    kept_labels = np.where(areas >= min_area)[0]
    if len(kept_labels) == 0:
        kept_labels = np.array([int(np.argmax(areas))])

    clean_mask = np.isin(labels, kept_labels)
    clean_mask = ndi.binary_fill_holes(clean_mask)

    ys, xs = np.where(clean_mask)
    if len(xs) == 0:
        raise ValueError('La mascara azul quedo vacia despues de limpiar componentes.')

    img_h, img_w = clean_mask.shape
    x1 = max(int(xs.min()) - padding, 0)
    y1 = max(int(ys.min()) - padding, 0)
    x2 = min(int(xs.max()) + padding, img_w - 1)
    y2 = min(int(ys.max()) + padding, img_h - 1)
    bbox = (x1, y1, x2, y2)

    overlay = overlay_mask_rgb(rgb_u8, clean_mask, color=(0, 220, 255), alpha=0.30)
    overlay = draw_rectangle_rgb(overlay, bbox, color=(255, 255, 0), thickness=4)

    info = {
        'bbox_xyxy': bbox,
        'bbox_width': x2 - x1 + 1,
        'bbox_height': y2 - y1 + 1,
        'mask_area_px': int(clean_mask.sum()),
        'kept_components': int(len(kept_labels)),
        'sat_min': sat_min,
        'blue_red_min': blue_red_min,
        'blue_excess_min': blue_excess_min,
    }
    return clean_mask, bbox, overlay, info


def open_hsi_and_detect_blue_box(hdr_path, specimen_id=None, show=True, **detect_kwargs):
    """Abre una HSI, genera pseudo-RGB y detecta la caja azul."""
    rgb_u8, hsi_info = hsi_pseudo_rgb_from_path(hdr_path)
    mask, bbox, overlay, box_info = detect_blue_box_from_rgb(rgb_u8, **detect_kwargs)

    result = {
        'specimen_id': specimen_id,
        'hdr_path': Path(hdr_path),
        'pseudo_rgb': rgb_u8,
        'blue_mask': mask,
        'overlay': overlay,
        'bbox_xyxy': bbox,
        'hsi_info': hsi_info,
        'box_info': box_info,
    }

    if show:
        plot_blue_box_detection(result)

    return result


def plot_blue_box_detection(result):
    specimen_id = result['specimen_id'] or result['hdr_path'].stem
    hsi_info = result['hsi_info']
    box_info = result['box_info']
    band_text = f"R={hsi_info.get('r_nm', hsi_info['r_idx']):.1f}, G={hsi_info.get('g_nm', hsi_info['g_idx']):.1f}, B={hsi_info.get('b_nm', hsi_info['b_idx']):.1f}"

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].imshow(result['pseudo_rgb'])
    axes[0].set_title(f'{specimen_id} - HSI pseudo-RGB\n{band_text}')

    axes[1].imshow(result['blue_mask'], cmap='gray')
    axes[1].set_title('Mascara caja azul')

    axes[2].imshow(result['overlay'])
    axes[2].set_title(
        'Caja azul detectada\n'
        f"bbox={box_info['bbox_xyxy']} | area={box_info['mask_area_px']} px"
    )

    for ax in axes:
        ax.axis('off')

    plt.show()



In [ ]:
def shrink_blue_box_detection(
    result,
    shrink_pixels=10,
    bbox_padding=0,
    kernel_shape='ellipse',
    show=True,
):
    """
    Reduce ligeramente el contorno de la caja azul ya detectada.

    No modifica el result original. Devuelve una copia con:
        - blue_mask_shrunk
        - overlay_shrunk
        - bbox_xyxy_shrunk
        - box_info_shrunk

    Parametros
    ----------
    result : dict
        Salida de open_hsi_and_detect_blue_box(...).
    shrink_pixels : int
        Pixeles aproximados que se erosiona la mascara azul hacia dentro.
    bbox_padding : int
        Margen extra que se vuelve a sumar al bbox despues de erosionar.
        Usa 0 para el ajuste mas estricto, o 5-15 si queda demasiado cerrado.
    kernel_shape : {'ellipse', 'rect'}
        Forma del kernel de erosion. 'ellipse' suele conservar mejor esquinas suaves.
    show : bool
        Si True, muestra comparacion entre deteccion original y reducida.
    """
    rgb_u8 = np.asarray(result['pseudo_rgb'], dtype=np.uint8)
    original_mask = np.asarray(result['blue_mask']).astype(bool)
    shrink_pixels = int(max(0, shrink_pixels))
    bbox_padding = int(max(0, bbox_padding))

    if shrink_pixels == 0:
        shrunk_mask = original_mask.copy()
    else:
        try:
            import cv2

            mask_u8 = original_mask.astype(np.uint8) * 255
            k = 2 * shrink_pixels + 1
            if kernel_shape == 'rect':
                kernel = cv2.getStructuringElement(cv2.MORPH_RECT, (k, k))
            else:
                kernel = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (k, k))
            shrunk_mask = cv2.erode(mask_u8, kernel, iterations=1) > 0
        except Exception:
            structure = np.ones((2 * shrink_pixels + 1, 2 * shrink_pixels + 1), dtype=bool)
            shrunk_mask = ndi.binary_erosion(original_mask, structure=structure)

    if not np.any(shrunk_mask):
        raise ValueError('La erosion ha vaciado la mascara. Prueba con menos shrink_pixels.')

    ys, xs = np.where(shrunk_mask)
    img_h, img_w = shrunk_mask.shape
    x1 = max(int(xs.min()) - bbox_padding, 0)
    y1 = max(int(ys.min()) - bbox_padding, 0)
    x2 = min(int(xs.max()) + bbox_padding, img_w - 1)
    y2 = min(int(ys.max()) + bbox_padding, img_h - 1)
    bbox = (x1, y1, x2, y2)

    overlay = overlay_mask_rgb(rgb_u8, shrunk_mask, color=(255, 180, 0), alpha=0.35)
    overlay = draw_rectangle_rgb(overlay, bbox, color=(255, 255, 0), thickness=4)

    adjusted = result.copy()
    adjusted['blue_mask_shrunk'] = shrunk_mask
    adjusted['overlay_shrunk'] = overlay
    adjusted['bbox_xyxy_shrunk'] = bbox
    adjusted['box_info_shrunk'] = {
        'bbox_xyxy': bbox,
        'bbox_width': x2 - x1 + 1,
        'bbox_height': y2 - y1 + 1,
        'mask_area_px': int(shrunk_mask.sum()),
        'shrink_pixels': shrink_pixels,
        'bbox_padding': bbox_padding,
        'kernel_shape': kernel_shape,
    }

    if show:
        plot_shrunk_blue_box_detection(adjusted)

    return adjusted


def plot_shrunk_blue_box_detection(adjusted_result):
    specimen_id = adjusted_result.get('specimen_id') or adjusted_result['hdr_path'].stem
    original_info = adjusted_result['box_info']
    shrunk_info = adjusted_result['box_info_shrunk']

    fig, axes = plt.subplots(1, 3, figsize=(18, 6), constrained_layout=True)

    axes[0].imshow(adjusted_result['overlay'])
    axes[0].set_title(
        'Deteccion original\n'
        f"bbox={original_info['bbox_xyxy']}"
    )

    axes[1].imshow(adjusted_result['blue_mask_shrunk'], cmap='gray')
    axes[1].set_title(
        f"Mascara reducida | shrink={shrunk_info['shrink_pixels']} px\n"
        f"area={shrunk_info['mask_area_px']} px"
    )

    axes[2].imshow(adjusted_result['overlay_shrunk'])
    axes[2].set_title(
        f'{specimen_id} - caja ajustada\n'
        f"bbox={shrunk_info['bbox_xyxy']}"
    )

    for ax in axes:
        ax.axis('off')

    plt.show()


def shrink_all_blue_box_detections(results, shrink_pixels=10, bbox_padding=0, show=True):
    """Aplica shrink_blue_box_detection a todos los results ya calculados."""
    adjusted_results = {}
    for specimen_id, result in results.items():
        print(f"\n=== {specimen_id} | shrink={shrink_pixels}px ===")
        adjusted = shrink_blue_box_detection(
            result,
            shrink_pixels=shrink_pixels,
            bbox_padding=bbox_padding,
            show=show,
        )
        adjusted_results[specimen_id] = adjusted
        print(adjusted['box_info_shrunk'])
    return adjusted_results




## Funcion de segmentacion H&E

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image


def _read_slidedat_sections(slidedat_path):
    sections = {}
    current = None
    for raw_line in Path(slidedat_path).read_text(errors='ignore').splitlines():
        line = raw_line.strip()
        if not line:
            continue
        if line.startswith('[') and line.endswith(']'):
            current = line[1:-1]
            sections[current] = {}
            continue
        if current is not None and '=' in line:
            key, value = line.split('=', 1)
            sections[current][key.strip()] = value.strip()
    return sections


def find_he_slidedat_ini(he_path):
    """Busca Slidedat.ini asociado a la ruta H&E indicada, sin cambiar de imagen."""
    he_path = Path(he_path)
    candidates = [
        he_path.parent / 'Slidedat.ini',
        he_path.parent.parent / 'Slidedat.ini',
    ]
    for candidate in candidates:
        if candidate.exists():
            return candidate
    return None


def _safe_float(value):
    try:
        return float(value)
    except (TypeError, ValueError):
        return None


def _safe_int(value):
    try:
        return int(round(float(value)))
    except (TypeError, ValueError):
        return None


def read_he_slidedat_info(he_path):
    """Lee la escala real de la H&E desde Slidedat.ini asociado a la ruta indicada."""
    he_path = Path(he_path)
    slidedat_path = find_he_slidedat_ini(he_path)
    if slidedat_path is None:
        raise FileNotFoundError(f'No se encontro Slidedat.ini junto a la H&E: {he_path}')

    sections = _read_slidedat_sections(slidedat_path)
    level_items = []
    for name, data in sections.items():
        if name.startswith('LAYER_0_LEVEL_') and name.endswith('_SECTION'):
            try:
                level = int(name.replace('LAYER_0_LEVEL_', '').replace('_SECTION', ''))
            except ValueError:
                continue
            mpp_x = _safe_float(data.get('MICROMETER_PER_PIXEL_X'))
            mpp_y = _safe_float(data.get('MICROMETER_PER_PIXEL_Y'))
            if mpp_x is not None and mpp_y is not None:
                level_items.append((level, mpp_x, mpp_y))

    if not level_items:
        raise ValueError(f'No se encontraron MICROMETER_PER_PIXEL_X/Y en {slidedat_path}')

    level_items = sorted(level_items, key=lambda x: x[0])
    level0, mpp0_x, mpp0_y = level_items[0]
    preview_level, preview_mpp_x, preview_mpp_y = level_items[-1]

    preview_img = Image.open(he_path)
    preview_w, preview_h = preview_img.size
    preview_img.close()

    # Estimacion del nivel 0 a partir del campo fisico de la preview.
    width0 = int(round(preview_w * preview_mpp_x / mpp0_x))
    height0 = int(round(preview_h * preview_mpp_y / mpp0_y))

    max_level = max(level for level, _, _ in level_items)
    level_downsamples = [None] * (max_level + 1)
    level_dimensions = [None] * (max_level + 1)
    level_mpps = [None] * (max_level + 1)

    for level, mpp_x, mpp_y in level_items:
        downsample_x = mpp_x / mpp0_x
        downsample_y = mpp_y / mpp0_y
        downsample = (downsample_x + downsample_y) / 2.0
        level_downsamples[level] = float(downsample)
        level_dimensions[level] = (
            int(round(width0 / downsample_x)),
            int(round(height0 / downsample_y)),
        )
        level_mpps[level] = (float(mpp_x), float(mpp_y))

    return {
        'slidedat_path': slidedat_path,
        'dimensions_level_0': (int(width0), int(height0)),
        'level_dimensions': level_dimensions,
        'level_downsamples': level_downsamples,
        'level_mpps': level_mpps,
        'mpp_x': float(mpp0_x),
        'mpp_y': float(mpp0_y),
        'preview_level': int(preview_level),
        'preview_mpp_x': float(preview_mpp_x),
        'preview_mpp_y': float(preview_mpp_y),
        'preview_dimensions': (int(preview_w), int(preview_h)),
        'objective': sections.get('GENERAL', {}).get('OBJECTIVE_MAGNIFICATION') or sections.get('GENERAL', {}).get('OBJECTIVE_NAME'),
    }


def extract_he_tissue_contour_image(
    slide_path,
    max_dim=1800,
    sat_thresh=8,
    val_thresh=253,
    od_thresh=0.025,
    min_area=500,
    open_kernel_size=3,
    close_kernel_size=45,
    grow_pixels=8,
    use_convex_hull=False,
    show_original=True,
    show_result=True,
    debug=False,
    return_mask=False,
    return_info=False
):
    """Segmenta la H&E usando siempre la imagen de la ruta indicada abierta con PIL."""
    slide_path = Path(slide_path)
    if not slide_path.exists():
        raise FileNotFoundError(f'No existe la H&E: {slide_path}')

    rgb_u8 = np.asarray(Image.open(slide_path).convert('RGB'), dtype=np.uint8)
    level_info = read_he_slidedat_info(slide_path)
    chosen_level = level_info['preview_level']
    level_w, level_h = rgb_u8.shape[1], rgb_u8.shape[0]

    hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)
    sat = hsv[:, :, 1]
    val = hsv[:, :, 2]

    rgb_float = rgb_u8.astype(np.float32) / 255.0
    od = -np.log(np.clip(rgb_float, 1e-6, 1.0))
    od_sum = od.sum(axis=2)

    mask_color = ((sat > sat_thresh) & (val < val_thresh)) | (od_sum > od_thresh)
    mask = mask_color.astype(np.uint8) * 255

    if open_kernel_size and open_kernel_size > 1:
        kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (open_kernel_size, open_kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel_open)

    if close_kernel_size and close_kernel_size > 1:
        kernel_close = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (close_kernel_size, close_kernel_size))
        mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel_close)

    num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask, connectivity=8)
    mask_clean = np.zeros_like(mask, dtype=np.uint8)
    if num_labels > 1:
        areas = stats[1:, cv2.CC_STAT_AREA]
        valid = np.where(areas >= min_area)[0]
        if len(valid) > 0:
            largest_label = 1 + valid[np.argmax(areas[valid])]
            mask_clean[labels == largest_label] = 255

    contours, _ = cv2.findContours(mask_clean, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    mask_final = np.zeros_like(mask_clean, dtype=np.uint8)
    if contours:
        tissue_contour = max(contours, key=cv2.contourArea)
        if use_convex_hull:
            tissue_contour = cv2.convexHull(tissue_contour)
        cv2.drawContours(mask_final, [tissue_contour], -1, 255, thickness=cv2.FILLED)

    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * grow_pixels + 1, 2 * grow_pixels + 1))
        mask_final = cv2.dilate(mask_final, kernel_grow, iterations=1)

    tissue_only_rgb = rgb_u8.copy()
    tissue_only_rgb[mask_final == 0] = 0

    if debug:
        plt.figure(figsize=(14, 8))
        for idx, (img, title, cmap) in enumerate([
            (rgb_u8, f'H&E original | PIL preview level={chosen_level}', None),
            (sat, 'Saturacion HSV', 'gray'),
            (val, 'Valor HSV', 'gray'),
            (mask, 'Mascara inicial', 'gray'),
            (mask_final, 'Mascara final por contorno', 'gray'),
            (tissue_only_rgb, 'Solo tejido', None),
        ], start=1):
            plt.subplot(2, 3, idx)
            plt.imshow(img, cmap=cmap)
            plt.title(title)
            plt.axis('off')
        plt.tight_layout()
        plt.show()
    elif show_original and show_result:
        plt.figure(figsize=(10, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f'H&E original | PIL preview level={chosen_level}')
        plt.axis('off')
        plt.subplot(1, 2, 2)
        plt.imshow(tissue_only_rgb)
        plt.title('Solo tejido')
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f'H&E original | PIL preview level={chosen_level}')
        plt.axis('off')
        plt.show()
    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(tissue_only_rgb)
        plt.title('Solo tejido')
        plt.axis('off')
        plt.show()

    outputs = [tissue_only_rgb]
    if return_mask:
        outputs.append(mask_final)
    if return_info:
        outputs.append({
            'chosen_level': int(chosen_level),
            'read_dimensions': (int(level_w), int(level_h)),
            'sat_thresh': sat_thresh,
            'val_thresh': val_thresh,
            'od_thresh': od_thresh,
            'grow_pixels': grow_pixels,
            'use_convex_hull': use_convex_hull,
            'mask_area_px': int(np.count_nonzero(mask_final)),
            'reader': 'PIL',
            'slidedat_path': level_info['slidedat_path'],
        })
    return outputs[0] if len(outputs) == 1 else tuple(outputs)


## Funciones H&E + caja HSI + medidas reales

In [ ]:
def read_he_preview(he_path):
    """Abre la H&E igual que en los notebooks anteriores: PIL.Image.open sobre el .mrxs."""
    he_path = Path(he_path)
    if not he_path.exists():
        raise FileNotFoundError(f'No existe la H&E: {he_path}')

    img = Image.open(he_path)
    img_format = img.format
    rgb = np.asarray(img.convert('RGB'), dtype=np.uint8)
    info = {
        'format': img_format,
        'shape': rgb.shape,
        'mode': 'RGB',
    }
    return rgb, info


def plot_single_rgb(rgb, title, figsize=(7, 6)):
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    ax.imshow(rgb)
    ax.set_title(title)
    ax.axis('off')
    plt.show()


def _find_mirax_mpp(properties, axis):
    suffix = f'MICROMETER_PER_PIXEL_{axis.upper()}'
    for key, value in properties.items():
        if key.upper().endswith(suffix):
            parsed = _safe_float(value)
            if parsed is not None:
                return parsed, key
    return None, None


def print_he_scale_info(he_path):
    """Imprime dimensiones y escala real usando siempre la ruta H&E indicada."""
    he_path = Path(he_path)
    if not he_path.exists():
        raise FileNotFoundError(f'No existe la H&E: {he_path}')

    try:
        slide = openslide.OpenSlide(str(he_path))
        try:
            props = dict(slide.properties)
            level_dimensions = list(slide.level_dimensions)
            level_downsamples = [float(v) for v in slide.level_downsamples]
            width_px, height_px = slide.dimensions

            mpp_x = _safe_float(props.get('openslide.mpp-x'))
            mpp_y = _safe_float(props.get('openslide.mpp-y'))
            if mpp_x is None:
                mpp_x, mpp_x_key = _find_mirax_mpp(props, 'x')
            else:
                mpp_x_key = 'openslide.mpp-x'
            if mpp_y is None:
                mpp_y, mpp_y_key = _find_mirax_mpp(props, 'y')
            else:
                mpp_y_key = 'openslide.mpp-y'

            reader = 'OpenSlide'
            objective = props.get('openslide.objective-power') or props.get('aperio.AppMag') or props.get('mirax.GENERAL.OBJECTIVE_MAGNIFICATION')
        finally:
            slide.close()
    except Exception:
        slidedat_info = read_he_slidedat_info(he_path)
        width_px, height_px = slidedat_info['dimensions_level_0']
        level_dimensions = slidedat_info['level_dimensions']
        level_downsamples = slidedat_info['level_downsamples']
        mpp_x = slidedat_info['mpp_x']
        mpp_y = slidedat_info['mpp_y']
        mpp_x_key = f"{slidedat_info['slidedat_path'].name} LAYER_0_LEVEL_0_SECTION"
        mpp_y_key = f"{slidedat_info['slidedat_path'].name} LAYER_0_LEVEL_0_SECTION"
        reader = 'PIL + Slidedat.ini'
        objective = slidedat_info.get('objective')

    print('2. Medidas H&E / escala real')
    print(f'Ruta H&E usada: {he_path}')
    print(f'Lector escala: {reader}')
    print(f'Dimensiones nivel 0: {width_px} x {height_px} px')
    print(f'Niveles disponibles: {level_dimensions}')
    print(f'Downsamples: {[round(v, 3) if v is not None else None for v in level_downsamples]}')

    if mpp_x is not None and mpp_y is not None:
        width_mm = width_px * mpp_x / 1000.0
        height_mm = height_px * mpp_y / 1000.0
        print(f'Micras/pixel X: {mpp_x:.6f} ({mpp_x_key})')
        print(f'Micras/pixel Y: {mpp_y:.6f} ({mpp_y_key})')
        print(f'Campo nivel 0 aprox.: {width_mm:.2f} mm x {height_mm:.2f} mm')
        print(f'Escala nivel 0: {10000.0 / mpp_x:.1f} px/cm horizontal | {10000.0 / mpp_y:.1f} px/cm vertical')
    else:
        print('No se encontraron metadatos micras/pixel suficientes para calcular escala real.')

    if objective is not None:
        print(f'Objetivo/magnificacion: {objective}')

    return {
        'openslide_path': he_path,
        'he_path': he_path,
        'reader': reader,
        'dimensions_level_0': (int(width_px), int(height_px)),
        'level_dimensions': level_dimensions,
        'level_downsamples': level_downsamples,
        'mpp_x': mpp_x,
        'mpp_y': mpp_y,
    }


def bbox_corners_xyxy(bbox):
    x1, y1, x2, y2 = bbox
    return np.array([
        [x1, y1],
        [x2, y1],
        [x2, y2],
        [x1, y2],
    ], dtype=np.float32)


def draw_dimension_line(ax, p1, p2, label, color, text_offset=(0, 0)):
    ax.annotate(
        '',
        xy=p2,
        xytext=p1,
        arrowprops=dict(arrowstyle='<->', color=color, lw=2.5, shrinkA=0, shrinkB=0),
    )
    mid = (np.asarray(p1, dtype=float) + np.asarray(p2, dtype=float)) / 2.0
    ax.text(
        mid[0] + text_offset[0],
        mid[1] + text_offset[1],
        label,
        color=color,
        fontsize=12,
        weight='bold',
        ha='center',
        va='center',
        bbox=dict(facecolor='black', alpha=0.55, edgecolor='none', pad=3),
    )


def plot_adjusted_box_with_real_measures(
    adjusted_result,
    length_cm=4.15,
    width_cm=2.85,
    title=None,
):
    """
    Dibuja la caja ajustada y asigna 4.15 cm al lado largo y 2.85 cm al lado corto.

    De momento se asume que la caja ajustada esta representada por su bbox horizontal/vertical.
    Si el lado horizontal es mayor que el vertical, 4.15 cm se asigna arriba/abajo.
    Si el vertical es mayor, 4.15 cm se asigna izquierda/derecha.
    """
    rgb = adjusted_result['pseudo_rgb']
    bbox = adjusted_result['bbox_xyxy_shrunk']
    x1, y1, x2, y2 = bbox
    box_w_px = x2 - x1 + 1
    box_h_px = y2 - y1 + 1

    horizontal_is_long = box_w_px >= box_h_px
    if horizontal_is_long:
        horizontal_cm = length_cm
        vertical_cm = width_cm
        horizontal_label = f'{length_cm:.2f} cm | lado largo'
        vertical_label = f'{width_cm:.2f} cm | lado corto'
    else:
        horizontal_cm = width_cm
        vertical_cm = length_cm
        horizontal_label = f'{width_cm:.2f} cm | lado corto'
        vertical_label = f'{length_cm:.2f} cm | lado largo'

    px_per_cm_horizontal = box_w_px / horizontal_cm
    px_per_cm_vertical = box_h_px / vertical_cm

    fig, ax = plt.subplots(figsize=(10, 8), constrained_layout=True)
    ax.imshow(rgb)

    overlay = adjusted_result['overlay_shrunk']
    ax.imshow(overlay, alpha=0.42)

    corners = bbox_corners_xyxy(bbox)
    closed = np.vstack([corners, corners[0]])
    ax.plot(closed[:, 0], closed[:, 1], color='yellow', linewidth=2.5)
    ax.scatter(corners[:, 0], corners[:, 1], color='yellow', s=35)

    offset = 32
    top_y = max(y1 - offset, 5)
    left_x = max(x1 - offset, 5)

    draw_dimension_line(
        ax,
        (x1, top_y),
        (x2, top_y),
        horizontal_label,
        color='lime',
        text_offset=(0, -18),
    )
    draw_dimension_line(
        ax,
        (left_x, y1),
        (left_x, y2),
        vertical_label,
        color='cyan',
        text_offset=(-48, 0),
    )

    ax.text(
        x1,
        min(y2 + 35, rgb.shape[0] - 5),
        f'Escala horizontal: {px_per_cm_horizontal:.1f} px/cm\nEscala vertical: {px_per_cm_vertical:.1f} px/cm',
        color='white',
        fontsize=11,
        ha='left',
        va='top',
        bbox=dict(facecolor='black', alpha=0.65, edgecolor='none', pad=4),
    )

    ax.set_title(title or '6. Caja azul ajustada con medidas reales')
    ax.axis('off')
    plt.show()

    return {
        'bbox_xyxy': bbox,
        'bbox_width_px': int(box_w_px),
        'bbox_height_px': int(box_h_px),
        'horizontal_cm': float(horizontal_cm),
        'vertical_cm': float(vertical_cm),
        'length_cm': float(length_cm),
        'width_cm': float(width_cm),
        'horizontal_is_long': bool(horizontal_is_long),
        'px_per_cm_horizontal': float(px_per_cm_horizontal),
        'px_per_cm_vertical': float(px_per_cm_vertical),
    }


def open_he_hsi_detect_and_measure_box(
    he_path,
    hsi_hdr_path,
    specimen_id='specimen',
    length_cm=4.15,
    width_cm=2.85,
    shrink_pixels=10,
    bbox_padding=0,
    show=True,
):
    """
    Flujo de salida:
        1. Plot H&E sola.
        2. Print de medidas/escala H&E.
        3. Segmentacion H&E usando 3_Segmentacion_H&E.ipynb.
        4. Plot HSI basica.
        5. Plot HSI con caja ajustada.
        6. Plot de asignacion de medidas reales.
    """
    he_rgb, he_info = read_he_preview(he_path)

    if show:
        plot_single_rgb(
            he_rgb,
            f'1. {specimen_id} - H&E\n{he_info["format"]} | {he_info["shape"][1]}x{he_info["shape"][0]} px',
            figsize=(6, 8),
        )

    he_scale_info = print_he_scale_info(he_path)

    he_tissue_rgb, he_tissue_mask, he_tissue_info = extract_he_tissue_contour_image(
        he_scale_info['openslide_path'],
        max_dim=1800,
        show_original=False,
        show_result=False,
        debug=False,
        return_mask=True,
        return_info=True,
    )

    if show:
        plot_single_rgb(
            he_tissue_rgb,
            f'3. {specimen_id} - H&E segmentada',
            figsize=(6, 8),
        )

    detection_result = open_hsi_and_detect_blue_box(
        hsi_hdr_path,
        specimen_id=specimen_id,
        show=False,
    )

    with io.StringIO() as _buffer, redirect_stdout(_buffer):
        adjusted_results = shrink_all_blue_box_detections(
            {specimen_id: detection_result},
            shrink_pixels=shrink_pixels,
            bbox_padding=bbox_padding,
            show=False,
        )
    adjusted_result = adjusted_results[specimen_id]

    if show:
        plot_single_rgb(
            detection_result['pseudo_rgb'],
            f'4. {specimen_id} - HSI basica\nR=651.9, G=548.4, B=449.5',
            figsize=(8, 6),
        )
        plot_single_rgb(
            adjusted_result['overlay_shrunk'],
            f'5. {specimen_id} - HSI | caja ajustada\nbbox={adjusted_result["bbox_xyxy_shrunk"]}',
            figsize=(8, 6),
        )

    measure_info = plot_adjusted_box_with_real_measures(
        adjusted_result,
        length_cm=length_cm,
        width_cm=width_cm,
        title=f'6. {specimen_id} - asignacion de medidas reales',
    )

    return {
        'specimen_id': specimen_id,
        'he_rgb': he_rgb,
        'he_info': he_info,
        'he_scale_info': he_scale_info,
        'he_tissue_rgb': he_tissue_rgb,
        'he_tissue_mask': he_tissue_mask,
        'he_tissue_info': he_tissue_info,
        'detection_result': detection_result,
        'adjusted_result': adjusted_result,
        'measure_info': measure_info,
    }


## Funciones para igualar escala real

In [ ]:
def mask_bbox_xyxy(mask, padding=0):
    """Obtiene bbox inclusiva (x1, y1, x2, y2) alrededor de una mascara."""
    mask_bool = np.asarray(mask) > 0
    ys, xs = np.where(mask_bool)
    if len(xs) == 0:
        raise ValueError('La mascara esta vacia; no se puede calcular el recorte.')

    h, w = mask_bool.shape[:2]
    x1 = max(int(xs.min()) - padding, 0)
    y1 = max(int(ys.min()) - padding, 0)
    x2 = min(int(xs.max()) + padding, w - 1)
    y2 = min(int(ys.max()) + padding, h - 1)
    return (x1, y1, x2, y2)


def crop_xyxy(img, bbox):
    """Recorta usando bbox inclusiva (x1, y1, x2, y2)."""
    x1, y1, x2, y2 = bbox
    return np.asarray(img)[y1:y2 + 1, x1:x2 + 1].copy()


def ensure_uint8_rgb(img):
    """Asegura una imagen RGB uint8 para visualizar/guardar."""
    arr = np.asarray(img)
    if arr.dtype == np.uint8:
        return arr
    arr = arr.astype(np.float32)
    finite = np.isfinite(arr)
    if not finite.any():
        return np.zeros((*arr.shape[:2], 3), dtype=np.uint8)
    arr = np.nan_to_num(arr, nan=0.0, posinf=0.0, neginf=0.0)
    if arr.max() <= 1.5:
        arr = arr * 255.0
    return np.clip(arr, 0, 255).astype(np.uint8)


def resize_to_target_px_per_cm(img, src_px_per_cm_x, src_px_per_cm_y, target_px_per_cm):
    """
    Reescala una imagen para que sus ejes X/Y queden en el mismo px/cm objetivo.

    Si src_px_per_cm_x != src_px_per_cm_y, corrige esa anisotropia reescalando cada eje
    con un factor distinto. El resultado queda con pixeles isotropicos a target_px_per_cm.
    """
    img = ensure_uint8_rgb(img)
    h, w = img.shape[:2]

    scale_x = float(target_px_per_cm) / float(src_px_per_cm_x)
    scale_y = float(target_px_per_cm) / float(src_px_per_cm_y)
    new_w = max(1, int(round(w * scale_x)))
    new_h = max(1, int(round(h * scale_y)))

    interpolation = cv2.INTER_AREA if scale_x < 1.0 and scale_y < 1.0 else cv2.INTER_LINEAR
    resized = cv2.resize(img, (new_w, new_h), interpolation=interpolation)
    return resized, {
        'original_shape': (int(h), int(w)),
        'scaled_shape': (int(new_h), int(new_w)),
        'scale_x': float(scale_x),
        'scale_y': float(scale_y),
        'src_px_per_cm_x': float(src_px_per_cm_x),
        'src_px_per_cm_y': float(src_px_per_cm_y),
        'target_px_per_cm': float(target_px_per_cm),
    }


def he_segmented_px_per_cm(output):
    """Calcula la escala de la H&E segmentada que devolvio extract_he_tissue_contour_image."""
    mpp_x = output['he_scale_info']['mpp_x']
    mpp_y = output['he_scale_info']['mpp_y']
    if mpp_x is None or mpp_y is None:
        raise ValueError('La H&E no tiene micras/pixel; no se puede igualar escala real.')

    chosen_level = output['he_tissue_info']['chosen_level']
    downsample = output['he_scale_info']['level_downsamples'][chosen_level]

    level_mpp_x = float(mpp_x) * float(downsample)
    level_mpp_y = float(mpp_y) * float(downsample)

    return {
        'chosen_level': int(chosen_level),
        'downsample': float(downsample),
        'mpp_x_at_level': float(level_mpp_x),
        'mpp_y_at_level': float(level_mpp_y),
        'px_per_cm_x': float(10000.0 / level_mpp_x),
        'px_per_cm_y': float(10000.0 / level_mpp_y),
    }


def hsi_box_px_per_cm(output):
    """Obtiene la escala de la HSI usando la caja azul ajustada de 4.15 x 2.85 cm."""
    measure = output['measure_info']
    return {
        'px_per_cm_x': float(measure['px_per_cm_horizontal']),
        'px_per_cm_y': float(measure['px_per_cm_vertical']),
        'horizontal_cm': float(measure['horizontal_cm']),
        'vertical_cm': float(measure['vertical_cm']),
        'bbox_xyxy': tuple(output['adjusted_result']['bbox_xyxy_shrunk']),
    }


def choose_target_px_per_cm(he_scale, hsi_scale, target='hsi'):
    """Elige la escala comun final."""
    if isinstance(target, (int, float)):
        return float(target)
    if target == 'hsi':
        return float((hsi_scale['px_per_cm_x'] + hsi_scale['px_per_cm_y']) / 2.0)
    if target == 'he':
        return float((he_scale['px_per_cm_x'] + he_scale['px_per_cm_y']) / 2.0)
    if target == 'min':
        return float(min(
            he_scale['px_per_cm_x'], he_scale['px_per_cm_y'],
            hsi_scale['px_per_cm_x'], hsi_scale['px_per_cm_y'],
        ))
    raise ValueError("target debe ser 'hsi', 'he', 'min' o un numero px/cm.")


def plot_equal_scale_pair(he_scaled, hsi_scaled, info):
    """Muestra los dos recortes ya reescalados. Las matrices resultantes comparten px/cm."""
    fig, axes = plt.subplots(1, 2, figsize=(12, 6), constrained_layout=True)

    axes[0].imshow(he_scaled)
    axes[0].set_title(
        f"H&E segmentada | misma escala\n"
        f"{he_scaled.shape[1]}x{he_scaled.shape[0]} px | {info['target_px_per_cm']:.1f} px/cm"
    )
    axes[0].axis('off')

    axes[1].imshow(hsi_scaled)
    axes[1].set_title(
        f"HSI caja/especimen | misma escala\n"
        f"{hsi_scaled.shape[1]}x{hsi_scaled.shape[0]} px | {info['target_px_per_cm']:.1f} px/cm"
    )
    axes[1].axis('off')

    plt.show()


def equalize_he_hsi_real_scale(
    output,
    target_px_per_cm='hsi',
    he_padding_px=20,
    hsi_padding_px=0,
    show=True,
):
    """
    Genera dos imagenes a la misma escala real:
        - H&E: recorte del tejido segmentado.
        - HSI: recorte calibrado por la caja azul ajustada.

    Importante: la escala fisica queda en las matrices devueltas. Es decir, si ambas
    imagenes se guardan o se comparan pixel a pixel, comparten el mismo px/cm.
    """
    he_scale = he_segmented_px_per_cm(output)
    hsi_scale = hsi_box_px_per_cm(output)
    target = choose_target_px_per_cm(he_scale, hsi_scale, target=target_px_per_cm)

    he_bbox = mask_bbox_xyxy(output['he_tissue_mask'], padding=he_padding_px)
    he_crop = crop_xyxy(output['he_tissue_rgb'], he_bbox)

    hsi_bbox = hsi_scale['bbox_xyxy']
    if hsi_padding_px:
        hsi_img = output['adjusted_result']['pseudo_rgb']
        x1, y1, x2, y2 = hsi_bbox
        h, w = hsi_img.shape[:2]
        hsi_bbox = (
            max(0, x1 - hsi_padding_px),
            max(0, y1 - hsi_padding_px),
            min(w - 1, x2 + hsi_padding_px),
            min(h - 1, y2 + hsi_padding_px),
        )
    hsi_crop = crop_xyxy(output['adjusted_result']['pseudo_rgb'], hsi_bbox)

    he_scaled, he_resize_info = resize_to_target_px_per_cm(
        he_crop,
        src_px_per_cm_x=he_scale['px_per_cm_x'],
        src_px_per_cm_y=he_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )
    hsi_scaled, hsi_resize_info = resize_to_target_px_per_cm(
        hsi_crop,
        src_px_per_cm_x=hsi_scale['px_per_cm_x'],
        src_px_per_cm_y=hsi_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )

    info = {
        'target_px_per_cm': float(target),
        'he_scale': he_scale,
        'hsi_scale': hsi_scale,
        'he_bbox_xyxy': he_bbox,
        'hsi_bbox_xyxy': hsi_bbox,
        'he_resize_info': he_resize_info,
        'hsi_resize_info': hsi_resize_info,
        'he_real_width_cm': float(he_crop.shape[1] / he_scale['px_per_cm_x']),
        'he_real_height_cm': float(he_crop.shape[0] / he_scale['px_per_cm_y']),
        'hsi_real_width_cm': float(hsi_crop.shape[1] / hsi_scale['px_per_cm_x']),
        'hsi_real_height_cm': float(hsi_crop.shape[0] / hsi_scale['px_per_cm_y']),
    }

    print('Escala comun final')
    print(f"  target: {target:.2f} px/cm")
    print(f"  H&E original: {he_scale['px_per_cm_x']:.2f} px/cm X | {he_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  HSI original: {hsi_scale['px_per_cm_x']:.2f} px/cm X | {hsi_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  H&E recorte real aprox.: {info['he_real_width_cm']:.2f} x {info['he_real_height_cm']:.2f} cm")
    print(f"  HSI recorte real aprox.: {info['hsi_real_width_cm']:.2f} x {info['hsi_real_height_cm']:.2f} cm")

    if show:
        plot_equal_scale_pair(he_scaled, hsi_scaled, info)

    return {
        'he_scaled_rgb': he_scaled,
        'hsi_scaled_rgb': hsi_scaled,
        'info': info,
    }


## Funciones de comprobacion en lienzo unico

In [ ]:
def print_equal_scale_diagnostics(same_scale_output):
    """Imprime comprobaciones numericas de escala para H&E y HSI ya reescaladas."""
    he = same_scale_output['he_scaled_rgb']
    hsi = same_scale_output['hsi_scaled_rgb']
    info = same_scale_output['info']
    target = info['target_px_per_cm']

    he_h, he_w = he.shape[:2]
    hsi_h, hsi_w = hsi.shape[:2]

    diagnostics = {
        'target_px_per_cm': float(target),
        'he_shape_px': (int(he_h), int(he_w)),
        'hsi_shape_px': (int(hsi_h), int(hsi_w)),
        'he_size_cm_from_pixels': (float(he_w / target), float(he_h / target)),
        'hsi_size_cm_from_pixels': (float(hsi_w / target), float(hsi_h / target)),
        'he_size_cm_before_resize': (info['he_real_width_cm'], info['he_real_height_cm']),
        'hsi_size_cm_before_resize': (info['hsi_real_width_cm'], info['hsi_real_height_cm']),
        'one_cm_bar_px': float(target),
    }

    print('Comprobacion 1: ambas imagenes finales usan el mismo px/cm')
    print(f"  Escala comun: {target:.2f} px/cm")
    print(f"  Barra de 1 cm debe medir: {target:.2f} px en ambas")
    print()
    print('Comprobacion 2: tama?o fisico calculado desde las matrices finales')
    print(f"  H&E final: {he_w}x{he_h} px -> {he_w / target:.2f} x {he_h / target:.2f} cm")
    print(f"  HSI final: {hsi_w}x{hsi_h} px -> {hsi_w / target:.2f} x {hsi_h / target:.2f} cm")
    print()
    print('Comprobacion 3: tama?o fisico antes/despues del resize')
    print(f"  H&E antes: {info['he_real_width_cm']:.2f} x {info['he_real_height_cm']:.2f} cm")
    print(f"  HSI antes: {info['hsi_real_width_cm']:.2f} x {info['hsi_real_height_cm']:.2f} cm")
    print()
    print('Interpretacion rapida')
    if he_w < hsi_w and he_h < hsi_h:
        print('  Numericamente la H&E es menor que la HSI en el lienzo final; si se ve mas grande, es efecto del subplot.')
    else:
        print('  Hay que revisar si el recorte H&E o la caja HSI estan representando la misma region anatomica.')

    return diagnostics


def add_scale_bar_to_rgb(img, px_per_cm, bar_cm=1.0, label=None, color=(255, 255, 0)):
    """A?ade una barra de escala dentro de una copia RGB."""
    out = ensure_uint8_rgb(img).copy()
    h, w = out.shape[:2]
    bar_px = max(1, int(round(px_per_cm * bar_cm)))
    bar_px = min(bar_px, max(1, w - 40))

    x1 = 20
    y1 = max(20, h - 30)
    x2 = x1 + bar_px
    cv2.line(out, (x1, y1), (x2, y1), color, thickness=4)
    cv2.line(out, (x1, y1 - 8), (x1, y1 + 8), color, thickness=3)
    cv2.line(out, (x2, y1 - 8), (x2, y1 + 8), color, thickness=3)

    text = label or f'{bar_cm:g} cm'
    cv2.putText(out, text, (x1, max(15, y1 - 14)), cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2, cv2.LINE_AA)
    return out


def plot_same_scale_on_one_canvas(same_scale_output, gap_px=80, background=255):
    """
    Pega H&E y HSI en un unico lienzo, respetando tama?os en pixeles.

    Esto evita el efecto visual de los subplots, donde cada axes se dibuja con tama?o parecido
    aunque las matrices tengan anchuras/alturas distintas.
    """
    he = same_scale_output['he_scaled_rgb']
    hsi = same_scale_output['hsi_scaled_rgb']
    target = same_scale_output['info']['target_px_per_cm']

    he_bar = add_scale_bar_to_rgb(he, target, bar_cm=1.0, label='1 cm')
    hsi_bar = add_scale_bar_to_rgb(hsi, target, bar_cm=1.0, label='1 cm')

    he_h, he_w = he_bar.shape[:2]
    hsi_h, hsi_w = hsi_bar.shape[:2]
    label_h = 42
    canvas_h = label_h + max(he_h, hsi_h)
    canvas_w = he_w + gap_px + hsi_w
    canvas = np.full((canvas_h, canvas_w, 3), background, dtype=np.uint8)

    he_y = label_h + (max(he_h, hsi_h) - he_h) // 2
    hsi_y = label_h + (max(he_h, hsi_h) - hsi_h) // 2
    he_x = 0
    hsi_x = he_w + gap_px

    canvas[he_y:he_y + he_h, he_x:he_x + he_w] = he_bar
    canvas[hsi_y:hsi_y + hsi_h, hsi_x:hsi_x + hsi_w] = hsi_bar

    cv2.putText(canvas, f'H&E {he_w}x{he_h}px', (he_x + 5, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 2, cv2.LINE_AA)
    cv2.putText(canvas, f'HSI {hsi_w}x{hsi_h}px', (hsi_x + 5, 26), cv2.FONT_HERSHEY_SIMPLEX, 0.75, (0, 0, 0), 2, cv2.LINE_AA)

    fig, ax = plt.subplots(figsize=(12, 6), constrained_layout=True)
    ax.imshow(canvas)
    ax.set_title(f'Comprobacion en lienzo unico | misma escala: {target:.1f} px/cm')
    ax.axis('off')
    plt.show()

    return canvas


## Funcion de segmentacion HSI

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
from spectral import open_image


def extract_specimen_only_from_hsi_path(
    hdr_path,
    r_nm=650,
    g_nm=550,
    b_nm=450,
    grow_pixels=5,
    show_original=True,
    show_result=True,
    return_mask=False,
    return_info=False,
):
    """
    Carga una imagen hiperespectral ENVI desde una ruta .hdr y devuelve una pseudo-RGB
    donde solo se ve el esp?cimen; el resto queda negro.

    La segmentacion HSI es adaptativa:
        1. Prueba el metodo original basado en caja azul / hueco interno.
        2. Prueba un metodo alternativo para casos donde en gris se ve mejor el tejido
           dentro de la bandeja oscura.
        3. Elige el metodo alternativo solo si el metodo original parece una mascara
           rectangular de la bandeja/caja, y la alternativa tiene calidad suficiente.
    """

    img = open_image(hdr_path)
    import warnings
    try:
        from spectral.io.spyfile import NaNValueWarning
        ignored_warnings = (NaNValueWarning,)
    except Exception:
        ignored_warnings = (Warning,)
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ignored_warnings)
        cube = np.asarray(img.load(), dtype=np.float32)

    wavelengths = np.array([float(w) for w in img.metadata["wavelength"]], dtype=np.float32)

    def find_nearest_band(wavelengths, target_nm):
        wavelengths_arr = np.asarray(wavelengths, dtype=float)
        return int(np.argmin(np.abs(wavelengths_arr - target_nm)))

    def robust_normalize(channel, p_low=2, p_high=98):
        ch = channel.astype(np.float32)
        ch = np.where(np.isfinite(ch), ch, np.nan)
        if np.all(np.isnan(ch)):
            return np.zeros_like(ch, dtype=np.float32)
        lo = np.nanpercentile(ch, p_low)
        hi = np.nanpercentile(ch, p_high)
        if not np.isfinite(lo) or not np.isfinite(hi) or hi <= lo:
            return np.zeros_like(ch, dtype=np.float32)
        ch = (ch - lo) / (hi - lo)
        ch = np.clip(ch, 0, 1)
        return np.nan_to_num(ch, nan=0.0, posinf=1.0, neginf=0.0).astype(np.float32)

    def largest_component(mask, min_area=1000, fill=False, avoid_border=False):
        mask_u8 = (np.asarray(mask) > 0).astype(np.uint8)
        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
        if num_labels <= 1:
            return np.zeros(mask_u8.shape, dtype=bool)

        img_h, img_w = mask_u8.shape
        candidates = []
        for label in range(1, num_labels):
            x, y, w, h, area = stats[label]
            if area < min_area:
                continue
            if avoid_border and (x <= 2 or y <= 2 or x + w >= img_w - 2 or y + h >= img_h - 2):
                continue
            candidates.append(label)

        if not candidates:
            candidates = [label for label in range(1, num_labels) if stats[label, cv2.CC_STAT_AREA] >= min_area]
        if not candidates:
            candidates = [1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))]

        cx_img = img_w / 2.0
        cy_img = img_h / 2.0

        def component_score(label):
            x, y, w, h, area = stats[label]
            cx = x + w / 2.0
            cy = y + h / 2.0
            centrality = 1.0 - min(0.8, np.hypot((cx - cx_img) / img_w, (cy - cy_img) / img_h) * 1.5)
            return float(area) * centrality

        best_label = max(candidates, key=component_score)
        selected = labels == best_label

        if fill:
            contours, _ = cv2.findContours(selected.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
            filled = np.zeros_like(mask_u8)
            if contours:
                cv2.drawContours(filled, [max(contours, key=cv2.contourArea)], -1, 255, thickness=cv2.FILLED)
            selected = filled > 0

        return selected

    def bbox_xyxy(mask, padding=0):
        ys, xs = np.where(np.asarray(mask) > 0)
        if len(xs) == 0:
            return None
        img_h, img_w = mask.shape[:2]
        return (
            max(int(xs.min()) - padding, 0),
            max(int(ys.min()) - padding, 0),
            min(int(xs.max()) + padding, img_w - 1),
            min(int(ys.max()) + padding, img_h - 1),
        )

    def mask_quality(mask):
        mask_bool = np.asarray(mask) > 0
        ys, xs = np.where(mask_bool)
        if len(xs) == 0:
            return {
                'area_px': 0,
                'bbox_area_px': 0,
                'fill_ratio': 0.0,
                'solidity': 0.0,
                'rectangularity': 0.0,
                'bbox_xyxy': None,
                'area_ratio_image': 0.0,
            }

        x1, x2 = int(xs.min()), int(xs.max())
        y1, y2 = int(ys.min()), int(ys.max())
        bbox_area = int((x2 - x1 + 1) * (y2 - y1 + 1))
        area = int(mask_bool.sum())
        fill_ratio = float(area / bbox_area) if bbox_area else 0.0

        contours, _ = cv2.findContours(mask_bool.astype(np.uint8) * 255, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        solidity = 0.0
        if contours:
            contour = max(contours, key=cv2.contourArea)
            hull = cv2.convexHull(contour)
            hull_area = cv2.contourArea(hull)
            contour_area = cv2.contourArea(contour)
            solidity = float(contour_area / hull_area) if hull_area > 0 else 0.0

        return {
            'area_px': area,
            'bbox_area_px': bbox_area,
            'fill_ratio': fill_ratio,
            'solidity': solidity,
            'rectangularity': float(fill_ratio * solidity),
            'bbox_xyxy': (x1, y1, x2, y2),
            'area_ratio_image': float(area / mask_bool.size),
        }

    def segment_by_blue_box_inner(rgb_u8):
        hsv = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2HSV)
        R = rgb_u8[:, :, 0].astype(np.int16)
        G = rgb_u8[:, :, 1].astype(np.int16)
        B = rgb_u8[:, :, 2].astype(np.int16)
        V = hsv[:, :, 2]

        lower_blue = np.array([85, 10, 40], dtype=np.uint8)
        upper_blue = np.array([125, 180, 255], dtype=np.uint8)
        mask_hsv = cv2.inRange(hsv, lower_blue, upper_blue)
        mask_dominance = ((B - R) > 8) & ((G - R) > 3) & (V > 50)
        mask_blue = mask_hsv & (mask_dominance.astype(np.uint8) * 255)

        kernel_open = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (3, 3))
        kernel_close = cv2.getStructuringElement(cv2.MORPH_RECT, (35, 35))
        kernel_dilate = cv2.getStructuringElement(cv2.MORPH_RECT, (15, 15))
        mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_OPEN, kernel_open)
        mask_blue = cv2.morphologyEx(mask_blue, cv2.MORPH_CLOSE, kernel_close)
        mask_blue = cv2.dilate(mask_blue, kernel_dilate, iterations=1)

        if np.count_nonzero(mask_blue) == 0:
            raise ValueError('No se detecto la caja azul. Revisa los umbrales HSV.')

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(mask_blue, connectivity=8)
        if num_labels <= 1:
            raise ValueError('No se encontro ningun componente azul valido.')

        largest_label = 1 + int(np.argmax(stats[1:, cv2.CC_STAT_AREA]))
        mask_box_blue = (labels == largest_label).astype(np.uint8) * 255

        contours, hierarchy = cv2.findContours(mask_box_blue, cv2.RETR_CCOMP, cv2.CHAIN_APPROX_SIMPLE)
        if hierarchy is None:
            raise ValueError('No se encontraron contornos en la mascara de la caja azul.')
        hierarchy = hierarchy[0]
        inner_contours = [cnt for cnt, h in zip(contours, hierarchy) if h[3] != -1]

        mask_specimen = np.zeros_like(mask_box_blue, dtype=np.uint8)
        source = 'blue_box_inner_contour'

        if inner_contours:
            specimen_contour = max(inner_contours, key=cv2.contourArea)
            cv2.drawContours(mask_specimen, [specimen_contour], -1, 255, thickness=cv2.FILLED)
        else:
            source = 'blue_box_color_fallback'
            x = stats[largest_label, cv2.CC_STAT_LEFT]
            y = stats[largest_label, cv2.CC_STAT_TOP]
            w = stats[largest_label, cv2.CC_STAT_WIDTH]
            h = stats[largest_label, cv2.CC_STAT_HEIGHT]
            pad = 5
            img_h, img_w = rgb_u8.shape[:2]
            x1 = max(int(x) - pad, 0)
            y1 = max(int(y) - pad, 0)
            x2 = min(int(x + w) + pad, img_w)
            y2 = min(int(y + h) + pad, img_h)

            crop_rgb = rgb_u8[y1:y2, x1:x2]
            crop_blue = mask_box_blue[y1:y2, x1:x2] > 0
            crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
            Sc = crop_hsv[:, :, 1]
            Vc = crop_hsv[:, :, 2]
            Rc = crop_rgb[:, :, 0].astype(np.int16)
            Gc = crop_rgb[:, :, 1].astype(np.int16)
            Bc = crop_rgb[:, :, 2].astype(np.int16)

            tissue_color = (((Rc > Bc + 8) | (Gc > Bc + 8) | (Rc > Gc + 8)) & (Sc > 25) & (Vc > 25) & (Vc < 245))
            not_blue_box = ~crop_blue
            not_white_background = ~((Sc < 35) & (Vc > 170))
            not_black_background = Vc > 25
            mask_crop = (tissue_color & not_blue_box & not_white_background & not_black_background).astype(np.uint8) * 255
            mask_crop = cv2.morphologyEx(mask_crop, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
            mask_crop = cv2.morphologyEx(mask_crop, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (17, 17)))
            mask_crop_clean = largest_component(mask_crop, min_area=1000, fill=True)
            mask_specimen[y1:y2, x1:x2] = mask_crop_clean.astype(np.uint8) * 255

        mask_specimen = cv2.morphologyEx(mask_specimen, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (5, 5)))
        return mask_specimen > 0, {'method': source, 'quality': mask_quality(mask_specimen)}

    def segment_by_gray_tray(rgb_u8):
        gray = cv2.cvtColor(rgb_u8, cv2.COLOR_RGB2GRAY)
        blur = cv2.GaussianBlur(gray, (5, 5), 0)
        dark = (blur < 55).astype(np.uint8) * 255
        dark = cv2.morphologyEx(dark, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_RECT, (31, 31)))
        tray_roi = largest_component(dark, min_area=20000, fill=True, avoid_border=False)
        roi_bbox = bbox_xyxy(tray_roi, padding=8)
        full_mask = np.zeros_like(gray, dtype=bool)
        if roi_bbox is None:
            return full_mask, {'method': 'gray_tray', 'ok': False, 'reason': 'empty_tray_roi', 'quality': mask_quality(full_mask)}

        x1, y1, x2, y2 = roi_bbox
        roi = tray_roi[y1:y2 + 1, x1:x2 + 1]
        crop_rgb = rgb_u8[y1:y2 + 1, x1:x2 + 1]
        crop_gray = gray[y1:y2 + 1, x1:x2 + 1]
        crop_hsv = cv2.cvtColor(crop_rgb, cv2.COLOR_RGB2HSV)
        S = crop_hsv[:, :, 1]
        V = crop_hsv[:, :, 2]
        R = crop_rgb[:, :, 0].astype(np.int16)
        G = crop_rgb[:, :, 1].astype(np.int16)
        B = crop_rgb[:, :, 2].astype(np.int16)

        vals = crop_gray[roi]
        if vals.size < 100:
            return full_mask, {'method': 'gray_tray', 'ok': False, 'reason': 'small_tray_roi', 'quality': mask_quality(full_mask)}

        threshold = max(15.0, float(np.percentile(vals, 20) + 0.12 * (np.percentile(vals, 98) - np.percentile(vals, 20))))
        chromatic_tissue = (S > 20) & (V > 15) & ((R > B + 2) | (G > B + 2) | (R > G + 2))
        brighter_than_tray = (crop_gray > threshold) & (V > 18)
        candidate = roi & (chromatic_tissue | brighter_than_tray)
        candidate &= ~(V > 225)
        candidate &= ~((S < 25) & (V > 150))

        candidate = cv2.morphologyEx(candidate.astype(np.uint8) * 255, cv2.MORPH_OPEN, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)))
        candidate = cv2.erode(candidate, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (9, 9)), iterations=1)
        candidate = largest_component(candidate, min_area=5000, fill=False, avoid_border=True)
        candidate = cv2.dilate(candidate.astype(np.uint8) * 255, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (11, 11)), iterations=1)
        candidate = cv2.morphologyEx(candidate, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (51, 51)))
        candidate = largest_component(candidate, min_area=5000, fill=True, avoid_border=False)
        candidate = cv2.morphologyEx(candidate.astype(np.uint8) * 255, cv2.MORPH_CLOSE, cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (41, 41))) > 0
        candidate = largest_component(candidate, min_area=5000, fill=True, avoid_border=False)

        full_mask[y1:y2 + 1, x1:x2 + 1] = candidate
        info = {
            'method': 'gray_tray',
            'ok': True,
            'roi_bbox_xyxy': roi_bbox,
            'threshold': float(threshold),
            'quality': mask_quality(full_mask),
        }
        return full_mask, info

    r_idx = find_nearest_band(wavelengths, r_nm)
    g_idx = find_nearest_band(wavelengths, g_nm)
    b_idx = find_nearest_band(wavelengths, b_nm)

    pseudo_rgb = np.stack([
        robust_normalize(cube[:, :, r_idx]),
        robust_normalize(cube[:, :, g_idx]),
        robust_normalize(cube[:, :, b_idx]),
    ], axis=-1)
    rgb_u8 = (np.clip(np.nan_to_num(pseudo_rgb, nan=0.0, posinf=1.0, neginf=0.0), 0, 1) * 255).astype(np.uint8)

    blue_mask, blue_info = segment_by_blue_box_inner(rgb_u8)
    gray_mask, gray_info = segment_by_gray_tray(rgb_u8)

    blue_q = blue_info['quality']
    gray_q = gray_info['quality']
    blue_looks_rectangular = (
        blue_info['method'] == 'blue_box_inner_contour'
        and blue_q['fill_ratio'] >= 0.82
        and blue_q['solidity'] >= 0.90
        and blue_q['area_ratio_image'] >= 0.12
    )
    gray_is_usable = (
        gray_info.get('ok', False)
        and gray_q['area_px'] >= 5000
        and gray_q['fill_ratio'] >= 0.25
        and gray_q['fill_ratio'] <= 0.92
        and gray_q['solidity'] >= 0.45
    )
    gray_is_better_shape = gray_q['rectangularity'] <= blue_q['rectangularity'] - 0.10

    if blue_looks_rectangular and gray_is_usable and gray_is_better_shape:
        chosen_mask = gray_mask
        chosen_info = gray_info.copy()
        chosen_info['selected_because'] = 'blue_mask_looked_rectangular'
    else:
        chosen_mask = blue_mask
        chosen_info = blue_info.copy()
        chosen_info['selected_because'] = 'blue_mask_quality_ok'

    chosen_info = {
        'selected_method': chosen_info.get('method'),
        'selected_because': chosen_info.get('selected_because'),
        'blue_box_candidate': blue_info,
        'gray_tray_candidate': gray_info,
        'wavelengths_nm': {
            'r': float(wavelengths[r_idx]),
            'g': float(wavelengths[g_idx]),
            'b': float(wavelengths[b_idx]),
        },
    }

    mask_specimen = chosen_mask.astype(np.uint8) * 255
    if grow_pixels > 0:
        kernel_grow = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (2 * grow_pixels + 1, 2 * grow_pixels + 1))
        mask_specimen = cv2.dilate(mask_specimen, kernel_grow, iterations=1)

    specimen_only_rgb = rgb_u8.copy()
    specimen_only_rgb[mask_specimen == 0] = 0

    if show_original and show_result:
        plt.figure(figsize=(12, 6))
        plt.subplot(1, 2, 1)
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis('off')
        plt.subplot(1, 2, 2)
        plt.imshow(specimen_only_rgb)
        plt.title(f"Solo especimen | metodo: {chosen_info['selected_method']}")
        plt.axis('off')
        plt.tight_layout()
        plt.show()
    elif show_original:
        plt.figure(figsize=(6, 6))
        plt.imshow(rgb_u8)
        plt.title(f"Pseudo-RGB original | R={wavelengths[r_idx]:.1f} nm, G={wavelengths[g_idx]:.1f} nm, B={wavelengths[b_idx]:.1f} nm")
        plt.axis('off')
        plt.show()
    elif show_result:
        plt.figure(figsize=(6, 6))
        plt.imshow(specimen_only_rgb)
        plt.title(f"Solo especimen | metodo: {chosen_info['selected_method']}")
        plt.axis('off')
        plt.show()

    if return_mask and return_info:
        return specimen_only_rgb, mask_specimen, chosen_info
    if return_mask:
        return specimen_only_rgb, mask_specimen
    if return_info:
        return specimen_only_rgb, chosen_info
    return specimen_only_rgb


## Igualar escala real usando HSI segmentada

In [ ]:
def equalize_he_hsi_segmented_real_scale(
    output,
    hsi_hdr_path,
    target_px_per_cm='hsi',
    he_padding_px=20,
    hsi_padding_px=10,
    hsi_grow_pixels=5,
    show=True,
):
    """
    Igual que equalize_he_hsi_real_scale, pero usando la HSI segmentada con la funcion
    de 2_Segmentacion_HSI.ipynb.

    La escala HSI se conserva desde la caja azul ajustada, no desde el bbox del especimen.
    """
    hsi_specimen_rgb, hsi_specimen_mask, hsi_segmentation_info = extract_specimen_only_from_hsi_path(
        str(hsi_hdr_path),
        grow_pixels=hsi_grow_pixels,
        show_original=False,
        show_result=False,
        return_mask=True,
        return_info=True,
    )

    he_scale = he_segmented_px_per_cm(output)
    hsi_scale = hsi_box_px_per_cm(output)
    target = choose_target_px_per_cm(he_scale, hsi_scale, target=target_px_per_cm)

    he_bbox = mask_bbox_xyxy(output['he_tissue_mask'], padding=he_padding_px)
    he_crop = crop_xyxy(output['he_tissue_rgb'], he_bbox)

    hsi_specimen_bbox = mask_bbox_xyxy(hsi_specimen_mask, padding=hsi_padding_px)
    hsi_specimen_crop = crop_xyxy(hsi_specimen_rgb, hsi_specimen_bbox)

    he_scaled, he_resize_info = resize_to_target_px_per_cm(
        he_crop,
        src_px_per_cm_x=he_scale['px_per_cm_x'],
        src_px_per_cm_y=he_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )
    hsi_segmented_scaled, hsi_resize_info = resize_to_target_px_per_cm(
        hsi_specimen_crop,
        src_px_per_cm_x=hsi_scale['px_per_cm_x'],
        src_px_per_cm_y=hsi_scale['px_per_cm_y'],
        target_px_per_cm=target,
    )

    info = {
        'target_px_per_cm': float(target),
        'he_scale': he_scale,
        'hsi_scale': hsi_scale,
        'he_bbox_xyxy': he_bbox,
        'hsi_specimen_bbox_xyxy': hsi_specimen_bbox,
        'he_resize_info': he_resize_info,
        'hsi_resize_info': hsi_resize_info,
        'he_real_width_cm': float(he_crop.shape[1] / he_scale['px_per_cm_x']),
        'he_real_height_cm': float(he_crop.shape[0] / he_scale['px_per_cm_y']),
        'hsi_segmented_real_width_cm': float(hsi_specimen_crop.shape[1] / hsi_scale['px_per_cm_x']),
        'hsi_segmented_real_height_cm': float(hsi_specimen_crop.shape[0] / hsi_scale['px_per_cm_y']),
        'hsi_mask_area_px': int(np.count_nonzero(hsi_specimen_mask)),
        'hsi_segmentation_method': hsi_segmentation_info.get('selected_method'),
        'hsi_segmentation_info': hsi_segmentation_info,
    }

    print('Escala comun final con HSI segmentada')
    print(f"  target: {target:.2f} px/cm")
    print(f"  H&E original: {he_scale['px_per_cm_x']:.2f} px/cm X | {he_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  HSI original: {hsi_scale['px_per_cm_x']:.2f} px/cm X | {hsi_scale['px_per_cm_y']:.2f} px/cm Y")
    print(f"  H&E recorte real aprox.: {info['he_real_width_cm']:.2f} x {info['he_real_height_cm']:.2f} cm")
    print(f"  HSI segmentada real aprox.: {info['hsi_segmented_real_width_cm']:.2f} x {info['hsi_segmented_real_height_cm']:.2f} cm")
    print(f"  HSI mascara area: {info['hsi_mask_area_px']} px")

    result = {
        'he_scaled_rgb': he_scaled,
        'hsi_scaled_rgb': hsi_segmented_scaled,
        'hsi_specimen_rgb': hsi_specimen_rgb,
        'hsi_specimen_mask': hsi_specimen_mask,
        'hsi_segmentation_info': hsi_segmentation_info,
        'info': info,
    }

    if show:
        plot_same_scale_on_one_canvas(result)

    return result


## Funcion principal de preprocesado

In [ ]:
def calculate_box_measure_info(adjusted_result, length_cm=4.15, width_cm=2.85):
    """Calcula la escala de la caja ajustada sin generar plot."""
    bbox = adjusted_result['bbox_xyxy_shrunk']
    x1, y1, x2, y2 = bbox
    box_w_px = x2 - x1 + 1
    box_h_px = y2 - y1 + 1

    horizontal_is_long = box_w_px >= box_h_px
    if horizontal_is_long:
        horizontal_cm = length_cm
        vertical_cm = width_cm
    else:
        horizontal_cm = width_cm
        vertical_cm = length_cm

    return {
        'bbox_xyxy': bbox,
        'bbox_width_px': int(box_w_px),
        'bbox_height_px': int(box_h_px),
        'horizontal_cm': float(horizontal_cm),
        'vertical_cm': float(vertical_cm),
        'length_cm': float(length_cm),
        'width_cm': float(width_cm),
        'horizontal_is_long': bool(horizontal_is_long),
        'px_per_cm_horizontal': float(box_w_px / horizontal_cm),
        'px_per_cm_vertical': float(box_h_px / vertical_cm),
    }


def run_silently(func, *args, **kwargs):
    """Ejecuta una funcion capturando prints intermedios."""
    with io.StringIO() as _buffer, redirect_stdout(_buffer):
        return func(*args, **kwargs)


def maybe_print_he_scale_info(he_path, show=True):
    """Obtiene la escala H&E; si show=False, no imprime nada."""
    if show:
        return print_he_scale_info(he_path)
    return run_silently(print_he_scale_info, he_path)


def plot_rgb_with_title(rgb, title, figsize=(7, 6)):
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    ax.imshow(rgb)
    ax.set_title(title)
    ax.axis('off')
    plt.show()


def plot_scaled_rgb_with_bar(rgb, px_per_cm, title, bar_cm=1.0, figsize=(7, 6)):
    rgb_bar = add_scale_bar_to_rgb(rgb, px_per_cm, bar_cm=bar_cm, label=f'{bar_cm:g} cm')
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    ax.imshow(rgb_bar)
    ax.set_title(f'{title}\n{rgb.shape[1]}x{rgb.shape[0]} px | {px_per_cm:.1f} px/cm')
    ax.axis('off')
    plt.show()


def plot_scaled_rgb_without_bar(rgb, px_per_cm, title, figsize=(7, 6)):
    fig, ax = plt.subplots(figsize=figsize, constrained_layout=True)
    ax.imshow(rgb)
    ax.set_title(f'{title}\n{rgb.shape[1]}x{rgb.shape[0]} px | {px_per_cm:.1f} px/cm')
    ax.axis('off')
    plt.show()


def build_output_for_scale(
    specimen_id,
    he_rgb,
    he_info,
    he_scale_info,
    he_tissue_rgb,
    he_tissue_mask,
    he_tissue_info,
    detection_result,
    adjusted_result,
    measure_info,
):
    """Agrupa resultados con la misma estructura usada por las funciones del notebook 8."""
    return {
        'specimen_id': specimen_id,
        'he_rgb': he_rgb,
        'he_info': he_info,
        'he_scale_info': he_scale_info,
        'he_tissue_rgb': he_tissue_rgb,
        'he_tissue_mask': he_tissue_mask,
        'he_tissue_info': he_tissue_info,
        'detection_result': detection_result,
        'adjusted_result': adjusted_result,
        'measure_info': measure_info,
    }


def preprocesar_he_hsi(
    he_path,
    hsi_hdr_path,
    specimen_id='specimen',
    length_cm=4.15,
    width_cm=2.85,
    shrink_pixels=10,
    bbox_padding=0,
    he_padding_px=20,
    hsi_padding_px=0,
    hsi_segmented_padding_px=10,
    hsi_grow_pixels=5,
    target_px_per_cm='hsi',
    show_he_basica=True,
    show_print_he_datos=True,
    show_hsi_basica=True,
    show_hsi_contorno_caja=True,
    show_hsi_contorno_medidas=True,
    show_lienzo_unico_caja=True,
    show_he_segmentada=True,
    show_hsi_segmentada=True,
    show_lienzo_unico_segmentadas=True,
    show_he_segmentada_escala=True,
    show_hsi_segmentada_escala=True,
    show_he_segmentada_escala_sin_barra=True,
    show_hsi_segmentada_escala_sin_barra=True,
):
    """
    Preprocesa una pareja H&E + HSI y permite mostrar/ocultar cada salida.

    Salidas controladas por flags:
        1. show_he_basica: H&E basica.
        2. show_print_he_datos: print de datos/escala H&E.
        3. show_hsi_basica: HSI basica.
        4. show_hsi_contorno_caja: HSI con contorno/caja ajustada.
        5. show_hsi_contorno_medidas: HSI con caja y medidas reales.
        6. show_lienzo_unico_caja: H&E segmentada vs caja HSI a misma escala.
        7. show_he_segmentada: H&E segmentada.
        8. show_hsi_segmentada: HSI segmentada.
        9. show_lienzo_unico_segmentadas: H&E segmentada vs HSI segmentada a misma escala.
        10. show_he_segmentada_escala: H&E segmentada reescalada con barra.
        11. show_hsi_segmentada_escala: HSI segmentada reescalada con barra.
        12. show_he_segmentada_escala_sin_barra: H&E segmentada reescalada sin barra amarilla.
        13. show_hsi_segmentada_escala_sin_barra: HSI segmentada reescalada sin barra amarilla.

    Devuelve un diccionario con todas las imagenes, mascaras, escalas y diagnosticos.
    """
    he_path = Path(he_path)
    hsi_hdr_path = Path(hsi_hdr_path)

    # 1. H&E basica
    he_rgb, he_info = read_he_preview(he_path)
    if show_he_basica:
        plot_rgb_with_title(
            he_rgb,
            f'1. {specimen_id} - H&E basica\n{he_info["format"]} | {he_info["shape"][1]}x{he_info["shape"][0]} px',
            figsize=(6, 8),
        )

    # 2. Datos H&E
    he_scale_info = maybe_print_he_scale_info(he_path, show=show_print_he_datos)

    # 3. HSI basica + deteccion de caja azul
    detection_result = open_hsi_and_detect_blue_box(
        hsi_hdr_path,
        specimen_id=specimen_id,
        show=False,
    )
    if show_hsi_basica:
        plot_rgb_with_title(
            detection_result['pseudo_rgb'],
            f'3. {specimen_id} - HSI basica\nR=651.9, G=548.4, B=449.5',
            figsize=(8, 6),
        )

    adjusted_result = run_silently(
        shrink_all_blue_box_detections,
        {specimen_id: detection_result},
        shrink_pixels=shrink_pixels,
        bbox_padding=bbox_padding,
        show=False,
    )[specimen_id]

    # 4. HSI contorno/caja ajustada
    if show_hsi_contorno_caja:
        plot_rgb_with_title(
            adjusted_result['overlay_shrunk'],
            f'4. {specimen_id} - HSI contorno caja\nbbox={adjusted_result["bbox_xyxy_shrunk"]}',
            figsize=(8, 6),
        )

    # 5. HSI contorno caja con medidas
    if show_hsi_contorno_medidas:
        measure_info = plot_adjusted_box_with_real_measures(
            adjusted_result,
            length_cm=length_cm,
            width_cm=width_cm,
            title=f'5. {specimen_id} - HSI caja con medidas reales',
        )
    else:
        measure_info = calculate_box_measure_info(
            adjusted_result,
            length_cm=length_cm,
            width_cm=width_cm,
        )

    # 7. H&E segmentada
    he_tissue_rgb, he_tissue_mask, he_tissue_info = extract_he_tissue_contour_image(
        he_scale_info['openslide_path'],
        max_dim=1800,
        show_original=False,
        show_result=False,
        debug=False,
        return_mask=True,
        return_info=True,
    )
    if show_he_segmentada:
        plot_rgb_with_title(
            he_tissue_rgb,
            f'7. {specimen_id} - H&E segmentada',
            figsize=(6, 8),
        )

    workflow_output = build_output_for_scale(
        specimen_id=specimen_id,
        he_rgb=he_rgb,
        he_info=he_info,
        he_scale_info=he_scale_info,
        he_tissue_rgb=he_tissue_rgb,
        he_tissue_mask=he_tissue_mask,
        he_tissue_info=he_tissue_info,
        detection_result=detection_result,
        adjusted_result=adjusted_result,
        measure_info=measure_info,
    )

    # 6. Comprobacion en lienzo unico usando caja HSI completa
    if show_lienzo_unico_caja:
        same_scale_box_output = equalize_he_hsi_real_scale(
            workflow_output,
            target_px_per_cm=target_px_per_cm,
            he_padding_px=he_padding_px,
            hsi_padding_px=hsi_padding_px,
            show=False,
        )
        diagnostics_box = print_equal_scale_diagnostics(same_scale_box_output)
        same_scale_box_canvas = plot_same_scale_on_one_canvas(same_scale_box_output)
    else:
        same_scale_box_output = run_silently(
            equalize_he_hsi_real_scale,
            workflow_output,
            target_px_per_cm=target_px_per_cm,
            he_padding_px=he_padding_px,
            hsi_padding_px=hsi_padding_px,
            show=False,
        )
        diagnostics_box = run_silently(print_equal_scale_diagnostics, same_scale_box_output)
        same_scale_box_canvas = None

    # 8. HSI segmentada
    hsi_specimen_rgb, hsi_specimen_mask = extract_specimen_only_from_hsi_path(
        str(hsi_hdr_path),
        grow_pixels=hsi_grow_pixels,
        show_original=False,
        show_result=False,
        return_mask=True,
    )
    if show_hsi_segmentada:
        plot_rgb_with_title(
            hsi_specimen_rgb,
            f'8. {specimen_id} - HSI segmentada',
            figsize=(8, 6),
        )

    # 9. H&E segmentada y HSI segmentada en la misma escala
    if show_lienzo_unico_segmentadas:
        same_scale_segmented_output = equalize_he_hsi_segmented_real_scale(
            workflow_output,
            hsi_hdr_path=hsi_hdr_path,
            target_px_per_cm=target_px_per_cm,
            he_padding_px=he_padding_px,
            hsi_padding_px=hsi_segmented_padding_px,
            hsi_grow_pixels=hsi_grow_pixels,
            show=False,
        )
        same_scale_segmented_output['info'].setdefault(
            'hsi_real_width_cm',
            same_scale_segmented_output['info'].get('hsi_segmented_real_width_cm'),
        )
        same_scale_segmented_output['info'].setdefault(
            'hsi_real_height_cm',
            same_scale_segmented_output['info'].get('hsi_segmented_real_height_cm'),
        )
        diagnostics_segmented = print_equal_scale_diagnostics(same_scale_segmented_output)
        same_scale_segmented_canvas = plot_same_scale_on_one_canvas(same_scale_segmented_output)
    else:
        same_scale_segmented_output = run_silently(
            equalize_he_hsi_segmented_real_scale,
            workflow_output,
            hsi_hdr_path=hsi_hdr_path,
            target_px_per_cm=target_px_per_cm,
            he_padding_px=he_padding_px,
            hsi_padding_px=hsi_segmented_padding_px,
            hsi_grow_pixels=hsi_grow_pixels,
            show=False,
        )
        same_scale_segmented_output['info'].setdefault(
            'hsi_real_width_cm',
            same_scale_segmented_output['info'].get('hsi_segmented_real_width_cm'),
        )
        same_scale_segmented_output['info'].setdefault(
            'hsi_real_height_cm',
            same_scale_segmented_output['info'].get('hsi_segmented_real_height_cm'),
        )
        diagnostics_segmented = run_silently(print_equal_scale_diagnostics, same_scale_segmented_output)
        same_scale_segmented_canvas = None

    common_px_per_cm = same_scale_segmented_output['info']['target_px_per_cm']

    # 10. H&E segmentada con escala real
    if show_he_segmentada_escala:
        plot_scaled_rgb_with_bar(
            same_scale_segmented_output['he_scaled_rgb'],
            common_px_per_cm,
            f'10. {specimen_id} - H&E segmentada con escala real',
            figsize=(6, 8),
        )

    # 11. HSI segmentada con escala real
    if show_hsi_segmentada_escala:
        plot_scaled_rgb_with_bar(
            same_scale_segmented_output['hsi_scaled_rgb'],
            common_px_per_cm,
            f'11. {specimen_id} - HSI segmentada con escala real',
            figsize=(8, 6),
        )

    # 12. H&E segmentada con escala real, sin barra de 1 cm
    if show_he_segmentada_escala_sin_barra:
        plot_scaled_rgb_without_bar(
            same_scale_segmented_output['he_scaled_rgb'],
            common_px_per_cm,
            f'12. {specimen_id} - H&E segmentada con escala real sin barra',
            figsize=(6, 8),
        )

    # 13. HSI segmentada con escala real, sin barra de 1 cm
    if show_hsi_segmentada_escala_sin_barra:
        plot_scaled_rgb_without_bar(
            same_scale_segmented_output['hsi_scaled_rgb'],
            common_px_per_cm,
            f'13. {specimen_id} - HSI segmentada con escala real sin barra',
            figsize=(8, 6),
        )

    return {
        'specimen_id': specimen_id,
        'he_rgb': he_rgb,
        'he_info': he_info,
        'he_scale_info': he_scale_info,
        'hsi_detection_result': detection_result,
        'hsi_adjusted_result': adjusted_result,
        'measure_info': measure_info,
        'he_tissue_rgb': he_tissue_rgb,
        'he_tissue_mask': he_tissue_mask,
        'he_tissue_info': he_tissue_info,
        'hsi_specimen_rgb': hsi_specimen_rgb,
        'hsi_specimen_mask': hsi_specimen_mask,
        'same_scale_box_output': same_scale_box_output,
        'same_scale_box_diagnostics': diagnostics_box,
        'same_scale_box_canvas': same_scale_box_canvas,
        'same_scale_segmented_output': same_scale_segmented_output,
        'same_scale_segmented_diagnostics': diagnostics_segmented,
        'same_scale_segmented_canvas': same_scale_segmented_canvas,
        'he_segmentada_escala_rgb': same_scale_segmented_output['he_scaled_rgb'],
        'hsi_segmentada_escala_rgb': same_scale_segmented_output['hsi_scaled_rgb'],
    }


## Ejemplo de uso

Cambia `EXAMPLE_ID` por `SB012`, `SB013`, `SB017`, `SB018`, `SB019` o `SB020`, o pasa directamente tus dos rutas a `preprocesar_he_hsi(...)`.

In [ ]:
EXAMPLE_ID = 'SB017'
example_paths = SPECIMENS[EXAMPLE_ID]

preprocess_output = preprocesar_he_hsi(
    he_path=example_paths['he_path'],
    hsi_hdr_path=example_paths['hsi_hdr_path'],
    specimen_id=EXAMPLE_ID,
    show_he_basica=True,
    show_print_he_datos=True,
    show_hsi_basica=True,
    show_hsi_contorno_caja=True,
    show_hsi_contorno_medidas=True,
    show_lienzo_unico_caja=True,
    show_he_segmentada=True,
    show_hsi_segmentada=True,
    show_lienzo_unico_segmentadas=True,
    show_he_segmentada_escala=True,
    show_hsi_segmentada_escala=True,
    show_he_segmentada_escala_sin_barra=True,
    show_hsi_segmentada_escala_sin_barra=True,
)

preprocess_output.keys()
